# processing large ensembles

In [1]:
import os, re
from collections import defaultdict

root = "/g/data/su28/MMLEAv2/atmosphere/monthly"
scenarios = ("ssp585", "rcp85")
member_re = re.compile(r"_(r\d+i\d+p\d+(?:f\d+)?)_")

def members_by_exp(var, model):
    d = os.path.join(root, var, model)
    hist, scen = set(), set()
    if not os.path.isdir(d):
        return hist, scen
    for f in os.listdir(d):
        m = member_re.search(f)
        if not m:
            continue
        mem = m.group(1)
        if "historical" in f:
            hist.add(mem)
        if any(s in f for s in scenarios):
            scen.add(mem)
    return hist, scen

models = sorted(set(os.listdir(os.path.join(root, "tas"))) |
                set(os.listdir(os.path.join(root, "pr"))))

print(f"{'Model':22s} {'tas hist':>8s} {'tas scen':>8s} {'pr hist':>8s} {'pr scen':>8s} {'usable':>7s}")
for model in models:
    th, ts = members_by_exp("tas", model)
    ph, ps = members_by_exp("pr", model)
    usable = th & ts & ph & ps          # continuous hist+scenario for both variables
    print(f"{model:22s} {len(th):8d} {len(ts):8d} {len(ph):8d} {len(ps):8d} {len(usable):7d}")

Model                  tas hist tas scen  pr hist  pr scen  usable
ACCESS-ESM1-5                40       40       40       40      40
CESM1-CAM5                   40       40       40       40      40
CESM2                         0        0        0        0       0
CSIRO-Mk3-6-0                30       30       30       30      30
CanESM2                      50       50       50       50      50
CanESM5                      65       50       65       50      50
E3SM-1-0                     14        0       14        0       0
E3SMv2                       21        0       21        0       0
EC-Earth3                    23       58       22       58       8
GFDL-CM3                     20       20       20       20      20
GFDL-ESM2M                   30       30       30       30      30
GFDL-SPEAR-MED               30       30       30       30      30
IPSL-CM6A-LR                 33        7       33        7       7
MIROC-ES2L                   31       10       31       10    

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
import numpy as np
import os
import xarray as xr
import xcdat as xc
# import matplotlib.pyplot as plt
# from matplotlib.colors import BoundaryNorm as BM
import pandas as pd
# import matplotlib as mpl
# import matplotlib.ticker as mticker
import netCDF4
# import cartopy.crs as ccrs
# import cartopy.feature as cfeature
# from cartopy.util import add_cyclic_point
# from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

In [4]:
# from dask.distributed import Client
# client = Client(n_workers=4, threads_per_worker=2)
# client

In [5]:
from scipy import stats

In [6]:
# mpl.rcParams['font.family'] = 'Droid Sans'
# mpl.rcParams['font.size'] = 12
# # Edit axes parameters
# mpl.rcParams['axes.linewidth'] = 1.5
# # Tick properties
# mpl.rcParams['xtick.major.size'] = 5
# mpl.rcParams['xtick.minor.size'] = 3
# mpl.rcParams['xtick.major.width'] = 1
# mpl.rcParams['xtick.direction'] = 'out'
# mpl.rcParams['ytick.major.size'] = 5
# mpl.rcParams['ytick.minor.size'] = 3
# mpl.rcParams['ytick.major.width'] = 1
# mpl.rcParams['ytick.direction'] = 'out'

In [7]:
from dask.diagnostics import ProgressBar

### Functions needed for the analysis

In [8]:
from functions import preproc_funcs as funcs

In [9]:
from functions import xr_lowess

In [10]:
from statsmodels.tsa.seasonal import STL


def loess1d(x, period):
    x_copy = x.copy()
    res = STL(x_copy, period=period).fit()
    return res.trend


def loess3d(x, dim, period):
    return xr.apply_ufunc(loess1d, x, input_core_dims=[[dim]], output_core_dims=[[dim]], kwargs=dict(period=period), vectorize=True, dask="parallelized")

In [11]:
import glob
import multiprocessing as mp

In [12]:
# Function to find the first file in each model's r1* directory
def find_all_files(pattern):
    all_paths = glob.glob(pattern)
    model_files = {}
    for path in all_paths:
        # Adjust the split indices based on your folder structure
        path_parts = path.split('/')
        # Assuming the model name is at index 7 (adjust if needed)
        # model_identifier = path_parts[7] + '_' + path_parts[9][:-1]
        model_identifier = path_parts[8] + '_' + path_parts[10][:-1]
        if model_identifier not in model_files:
            model_files[model_identifier] = '/'.join(path_parts[:-1]) + '/*.nc'  # Store only the first file for each model
    return model_files


# Function to find the first file in each model's r1* directory
def find_all_files_extended(pattern):
    all_paths = glob.glob(pattern)
    model_files = {}
    for path in all_paths:
        # Adjust the split indices based on your folder structure
        path_parts = path.split('/')
        # Assuming the model name is at index 7 (adjust if needed)
        model_identifier = path_parts[8] + '_' + path_parts[10][:-1]
        if model_identifier not in model_files:
            model_files[model_identifier] = '/'.join(path_parts[:-1]) + '/*.nc'  # Store only the first file for each model
    return model_files



## find the files for a single model

In [13]:
model = 'MPI-ESM1-2-LR'

In [14]:
import re

In [29]:
## cmip6
ts_pattern_hist = f'/g/data/su28/MMLEAv2/atmosphere/monthly/tas/{model}/tas_Amon_{model}_historical*.nc'
ts_pattern_ssp5 = f'/g/data/su28/MMLEAv2/atmosphere/monthly/tas/{model}/tas_Amon_{model}_ssp585*.nc'
# ts_pattern_ssp3 = f'/g/data/su28/MMLEAv2/atmosphere/monthly/tas/{model}/tas_Amon_{model}*_historical_ssp370*.nc'

## cmip5
# ts_pattern_rcp85 = f'/g/data/su28/MMLEAv2/atmosphere/monthly/tas/{model}/tas_Amon_{model}_historical_rcp85_*.nc'

In [30]:
# ## cmip6
pr_pattern_hist = f'/g/data/su28/MMLEAv2/atmosphere/monthly/pr/{model}/pr_Amon_{model}_historical*.nc'
pr_pattern_ssp5 = f'/g/data/su28/MMLEAv2/atmosphere/monthly/pr/{model}/pr_Amon_{model}_ssp585*.nc'
# pr_pattern_ssp3 = f'/g/data/su28/MMLEAv2/atmosphere/monthly/pr/{model}/pr_Amon_{model}*_historical_ssp370*.nc'

## cmip5
# pr_pattern_rcp85 = f'/g/data/su28/MMLEAv2/atmosphere/monthly/pr/{model}/pr_Amon_{model}_historical_rcp85_*.nc'

In [31]:
ts_hist_files = sorted(glob.glob(ts_pattern_hist))
ts_ssp5_files = sorted(glob.glob(ts_pattern_ssp5))

# ts_rcp85_files = sorted(glob.glob(ts_pattern_rcp85))

In [32]:
pr_hist_files = sorted(glob.glob(pr_pattern_hist))
pr_ssp5_files = sorted(glob.glob(pr_pattern_ssp5))

# pr_rcp85_files = sorted(glob.glob(pr_pattern_rcp85))

In [33]:
# del ts_hist_files[3:6]
# del pr_hist_files[3:6]

In [34]:
# assert len(pr_hist_files) == len(pr_ssp5_files)

In [35]:
len(ts_ssp5_files), ts_ssp5_files

(30,
 ['/g/data/su28/MMLEAv2/atmosphere/monthly/tas/MPI-ESM1-2-LR/tas_Amon_MPI-ESM1-2-LR_ssp585_r10i1p1f1_g025_201501-210012.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/tas/MPI-ESM1-2-LR/tas_Amon_MPI-ESM1-2-LR_ssp585_r11i1p1f1_g025_201501-210012.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/tas/MPI-ESM1-2-LR/tas_Amon_MPI-ESM1-2-LR_ssp585_r12i1p1f1_g025_201501-210012.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/tas/MPI-ESM1-2-LR/tas_Amon_MPI-ESM1-2-LR_ssp585_r13i1p1f1_g025_201501-210012.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/tas/MPI-ESM1-2-LR/tas_Amon_MPI-ESM1-2-LR_ssp585_r14i1p1f1_g025_201501-210012.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/tas/MPI-ESM1-2-LR/tas_Amon_MPI-ESM1-2-LR_ssp585_r15i1p1f1_g025_201501-210012.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/tas/MPI-ESM1-2-LR/tas_Amon_MPI-ESM1-2-LR_ssp585_r16i1p1f1_g025_201501-210012.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/tas/MPI-ESM1-2-LR/tas_Amon_MPI-ESM1-2-LR_ssp585_r17i1p1f1_g025_201501-210012.

In [36]:
len(ts_hist_files), ts_hist_files

(50,
 ['/g/data/su28/MMLEAv2/atmosphere/monthly/tas/MPI-ESM1-2-LR/tas_Amon_MPI-ESM1-2-LR_historical_r10i1p1f1_g025_185001-201412.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/tas/MPI-ESM1-2-LR/tas_Amon_MPI-ESM1-2-LR_historical_r11i1p1f1_g025_185001-201412.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/tas/MPI-ESM1-2-LR/tas_Amon_MPI-ESM1-2-LR_historical_r12i1p1f1_g025_185001-201412.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/tas/MPI-ESM1-2-LR/tas_Amon_MPI-ESM1-2-LR_historical_r13i1p1f1_g025_185001-201412.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/tas/MPI-ESM1-2-LR/tas_Amon_MPI-ESM1-2-LR_historical_r14i1p1f1_g025_185001-201412.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/tas/MPI-ESM1-2-LR/tas_Amon_MPI-ESM1-2-LR_historical_r15i1p1f1_g025_185001-201412.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/tas/MPI-ESM1-2-LR/tas_Amon_MPI-ESM1-2-LR_historical_r16i1p1f1_g025_185001-201412.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/tas/MPI-ESM1-2-LR/tas_Amon_MPI-ESM1-2-LR_historic

In [37]:
import re, os

def member(f):
    return re.search(r"_(r\d+i\d+p\d+f\d+)_", os.path.basename(f)).group(1)

def by_member(files):
    return {member(f): f for f in files}

ts_hist, ts_ssp5 = by_member(ts_hist_files), by_member(ts_ssp5_files)
pr_hist, pr_ssp5 = by_member(pr_hist_files), by_member(pr_ssp5_files)

# members present in all four
common = set(ts_hist) & set(ts_ssp5) & set(pr_hist) & set(pr_ssp5)
common = sorted(common, key=lambda m: int(re.match(r"r(\d+)", m).group(1)))  # r1, r2, ..., r30

ts_hist_files = [ts_hist[m] for m in common]
ts_ssp5_files = [ts_ssp5[m] for m in common]
pr_hist_files = [pr_hist[m] for m in common]
pr_ssp5_files = [pr_ssp5[m] for m in common]

print(len(common), common)

30 ['r1i1p1f1', 'r2i1p1f1', 'r3i1p1f1', 'r4i1p1f1', 'r5i1p1f1', 'r6i1p1f1', 'r7i1p1f1', 'r8i1p1f1', 'r9i1p1f1', 'r10i1p1f1', 'r11i1p1f1', 'r12i1p1f1', 'r13i1p1f1', 'r14i1p1f1', 'r15i1p1f1', 'r16i1p1f1', 'r17i1p1f1', 'r18i1p1f1', 'r19i1p1f1', 'r20i1p1f1', 'r21i1p1f1', 'r22i1p1f1', 'r23i1p1f1', 'r24i1p1f1', 'r25i1p1f1', 'r26i1p1f1', 'r27i1p1f1', 'r28i1p1f1', 'r29i1p1f1', 'r30i1p1f1']


In [43]:
# len(pr_ssp5_files), pr_ssp5_files

In [44]:
def open_member_cmip6(hist_file, ssp_file, ens_idx):
    ds_hist = xr.open_mfdataset(hist_file, use_cftime=True)
    ds_ssp = xr.open_mfdataset(ssp_file, use_cftime=True)

    # concat in time
    ds = xr.concat([ds_hist, ds_ssp], dim='time')

    # enforce monotonic time and drop overlaps if any
    ds = ds.sortby('time')
    _, index = np.unique(ds.time, return_index=True)
    ds = ds.isel(time = index)

    # add_ensemble dimension
    ds = ds.assign_coords(model = ens_idx).expand_dims('model')

    return ds


def open_member_cmip6_alternate(ssp_file, ens_idx):
    ds = xr.open_mfdataset(ssp_file, use_cftime=True)

    # concat in time

    # enforce monotonic time and drop overlaps if any
    ds = ds.sortby('time')
    _, index = np.unique(ds.time, return_index=True)
    ds = ds.isel(time = index)

    # add_ensemble dimension
    ds = ds.assign_coords(model = ens_idx).expand_dims('model')

    return ds


def open_member_cmip5(rcp_file, ens_idx):
    ds = xr.open_mfdataset(rcp_file, use_cftime=True)

    # enforce monotonic time and drop overlaps if any
    ds = ds.sortby('time')
    _, index = np.unique(ds.time, return_index=True)
    ds = ds.isel(time = index)

    # add_ensemble dimension
    ds = ds.assign_coords(model = ens_idx).expand_dims('model')

    return ds

In [45]:
members = []
for i, (hf, sf) in enumerate(zip(ts_hist_files, ts_ssp5_files)):
    members.append(open_member_cmip6(hf, sf, i))


# members = []
# for i, rf in enumerate(ts_ssp5_files):
#     members.append(open_member_cmip6_alternate(rf, i))

In [46]:
ds = xr.concat(members, dim='model')
ds

<xarray.Dataset> Size: 4GB
Dimensions:    (model: 30, time: 3012, bnds: 2, lat: 72, lon: 144)
Coordinates:
  * model      (model) int64 240B 0 1 2 3 4 5 6 7 8 ... 22 23 24 25 26 27 28 29
  * time       (time) object 24kB 1850-01-16 12:00:00 ... 2100-12-16 12:00:00
  * lat        (lat) float64 576B -88.75 -86.25 -83.75 ... 83.75 86.25 88.75
  * lon        (lon) float64 1kB 1.25 3.75 6.25 8.75 ... 351.2 353.8 356.2 358.8
    height     float64 8B 2.0
Dimensions without coordinates: bnds
Data variables:
    time_bnds  (model, time, bnds) object 1MB dask.array<chunksize=(1, 1, 2), meta=np.ndarray>
    tas        (model, time, lat, lon) float32 4GB dask.array<chunksize=(1, 1, 72, 144), meta=np.ndarray>
Attributes: (12/49)
    CDI:                    Climate Data Interface version 2.2.4 (https://mpi...
    Conventions:            CF-1.7 CMIP-6.2
    source:                 MPI-ESM1.2-LR (2017): \naerosol: none, prescribed...
    institution:            Max Planck Institute for Meteorology, Hamburg 201...
    activity_id:            CMIP
    branch_method:          standard
    ...                     ...
    variable_id:            tas
    variant_label:          r1i1p1f1
    license:                CMIP6 model data produced by MPI-M is licensed un...
    cmor_version:           3.5.0
    tracking_id:            hdl:21.14100/6b679cba-17b8-45eb-90dc-23d170c1998c
    CDO:                    Climate Data Operators version 2.2.2 (https://mpi...

In [47]:
tas = ds["tas"]
# tas = tas.sel(time = slice('1850-01-01', '2100-12-31'))
tas = tas.chunk(dict(model=8, time=120, lat=50, lon=50))

In [48]:
tas_ann = tas.resample(time = 'AS-JUN').mean('time')

In [49]:
with ProgressBar():
    tas_ann_loaded = tas_ann.load()

[########################################] | 100% Completed | 489.89 s


In [50]:
members = []
for i, (hf, sf) in enumerate(zip(pr_hist_files, pr_ssp5_files)):
    members.append(open_member_cmip6(hf, sf, i))
    

# members = []
# for i, rf in enumerate(pr_ssp3_files):
#     members.append(open_member_cmip6_alternate(rf, i))

In [51]:
ds = xr.concat(members, dim='model')
ds

<xarray.Dataset> Size: 4GB
Dimensions:    (model: 30, time: 3012, bnds: 2, lat: 72, lon: 144)
Coordinates:
  * model      (model) int64 240B 0 1 2 3 4 5 6 7 8 ... 22 23 24 25 26 27 28 29
  * time       (time) object 24kB 1850-01-16 12:00:00 ... 2100-12-16 12:00:00
  * lat        (lat) float64 576B -88.75 -86.25 -83.75 ... 83.75 86.25 88.75
  * lon        (lon) float64 1kB 1.25 3.75 6.25 8.75 ... 351.2 353.8 356.2 358.8
Dimensions without coordinates: bnds
Data variables:
    time_bnds  (model, time, bnds) object 1MB dask.array<chunksize=(1, 1, 2), meta=np.ndarray>
    pr         (model, time, lat, lon) float32 4GB dask.array<chunksize=(1, 1, 72, 144), meta=np.ndarray>
Attributes: (12/49)
    CDI:                    Climate Data Interface version 2.2.4 (https://mpi...
    Conventions:            CF-1.7 CMIP-6.2
    source:                 MPI-ESM1.2-LR (2017): \naerosol: none, prescribed...
    institution:            Max Planck Institute for Meteorology, Hamburg 201...
    activity_id:            CMIP
    branch_method:          standard
    ...                     ...
    variable_id:            pr
    variant_label:          r1i1p1f1
    license:                CMIP6 model data produced by MPI-M is licensed un...
    cmor_version:           3.5.0
    tracking_id:            hdl:21.14100/63a418ca-fb43-418a-affc-a6000c95aa9c
    CDO:                    Climate Data Operators version 2.2.2 (https://mpi...

In [52]:
pr = ds["pr"]
# pr = pr.sel(time = slice('1850-01-01', '2100-12-31'))
pr = pr.chunk(dict(model=8, time=120, lat=50, lon=50))

In [53]:
pr_ann = pr.resample(time = 'AS-JUN').mean('time')

In [54]:
with ProgressBar():
    pr_ann_loaded = pr_ann.load()

[#                                       ] | 3% Completed | 22.48 sms

[########################################] | 100% Completed | 491.93 s


In [55]:
pr_ann_loaded

<xarray.DataArray 'pr' (model: 30, time: 252, lat: 72, lon: 144)> Size: 314MB
array([[[[1.92007519e-06, 1.94544691e-06, 1.97997315e-06, ...,
          1.85651822e-06, 1.88637898e-06, 1.90429125e-06],
         [1.55195869e-06, 1.51337258e-06, 1.58340811e-06, ...,
          1.50584128e-06, 1.53137557e-06, 1.56358715e-06],
         [1.41179191e-06, 1.24364476e-06, 1.32746231e-06, ...,
          1.41894418e-06, 1.36264111e-06, 1.37220445e-06],
         ...,
         [9.33332376e-06, 8.84926885e-06, 8.44902934e-06, ...,
          1.05581375e-05, 1.01238893e-05, 9.76580486e-06],
         [6.38735492e-06, 6.27692953e-06, 5.97278631e-06, ...,
          6.73682098e-06, 6.67468566e-06, 6.57171040e-06],
         [4.05595983e-06, 4.12853433e-06, 4.16503372e-06, ...,
          3.95338157e-06, 3.99825967e-06, 4.00379304e-06]],

        [[1.94334393e-06, 1.96866745e-06, 1.96656947e-06, ...,
          1.89346247e-06, 1.92333255e-06, 1.94290533e-06],
         [2.32036541e-06, 2.44810826e-06, 2.28730255e-06, ...,
          2.30270507e-06, 2.34315189e-06, 2.43530781e-06],
         [2.56024327e-06, 2.47631874e-06, 2.25575309e-06, ...,
          2.99612839e-06, 2.84395082e-06, 2.82664973e-06],
...
          1.33550457e-05, 1.27402345e-05, 1.24224898e-05],
         [1.27712983e-05, 1.28105266e-05, 1.29782084e-05, ...,
          1.25017468e-05, 1.24572607e-05, 1.25540946e-05],
         [8.80453354e-06, 8.90641604e-06, 8.99722636e-06, ...,
          8.66347364e-06, 8.67856579e-06, 8.75638852e-06]],

        [[1.81092184e-06, 1.81980590e-06, 1.84280748e-06, ...,
          1.76040214e-06, 1.75083903e-06, 1.76510298e-06],
         [2.00436762e-06, 2.00152886e-06, 1.98521002e-06, ...,
          1.92346374e-06, 1.85218937e-06, 1.88681406e-06],
         [1.99046144e-06, 1.93769120e-06, 1.83302529e-06, ...,
          2.12252485e-06, 2.12460213e-06, 2.05120705e-06],
         ...,
         [1.95653738e-05, 1.88458762e-05, 1.81824598e-05, ...,
          2.24150226e-05, 2.13211588e-05, 2.03011605e-05],
         [1.68825118e-05, 1.69622781e-05, 1.72152759e-05, ...,
          1.69263058e-05, 1.69913983e-05, 1.68651040e-05],
         [1.03564480e-05, 1.04534092e-05, 1.05807221e-05, ...,
          9.97505958e-06, 1.00707284e-05, 1.02062613e-05]]]],
      shape=(30, 252, 72, 144), dtype=float32)
Coordinates:
  * model    (model) int64 240B 0 1 2 3 4 5 6 7 8 ... 21 22 23 24 25 26 27 28 29
  * time     (time) object 2kB 1849-06-01 00:00:00 ... 2100-06-01 00:00:00
  * lat      (lat) float64 576B -88.75 -86.25 -83.75 ... 83.75 86.25 88.75
  * lon      (lon) float64 1kB 1.25 3.75 6.25 8.75 ... 351.2 353.8 356.2 358.8
Attributes:
    standard_name:  precipitation_flux
    long_name:      Precipitation
    units:          kg m-2 s-1
    comment:        includes both liquid and solid phases
    original_name:  pr
    cell_methods:   area: time: mean
    cell_measures:  area: areacella
    history:        2019-09-11T14:13:17Z altered by CMOR: replaced missing va...

In [56]:
tas_ann_loaded.to_netcdf(f'/g/data/if69/as8561/data/enso_tele_robustness/smiles/{model}_ts.nc')
pr_ann_loaded.to_netcdf(f'/g/data/if69/as8561/data/enso_tele_robustness/smiles/{model}_pr.nc')

In [57]:
# tas_anom = funcs.calc_anom(tas_ann_loaded, tas_ann_loaded.sel(time = slice('1960', '1990')))
# tas_det = funcs.detrend_rolling_window(tas_anom, window_size=15)

In [58]:
# pr_det = funcs.calc_anom(pr_ann_loaded, pr_ann_loaded.sel(time = slice('1960', '1990')))
# pr_det = funcs.detrend_rolling_window(pr_det, window_size=15)
# pr_std = pr_det/pr_det.std('time')